In [1]:
# Allowing dynamic links, hot-reload, to our custom modules
%load_ext autoreload
%autoreload 2

In [2]:
import os, re, sys, traceback, csv, random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sentencepiece as spm

from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data

os.getcwd()

'c:\\Users\\shrey\\Downloads\\dl-gen-lyrics\\sandbox'

In [3]:
import generator_core as core
from aspect_teal import Teal
import aspect_teal as teal
from aspect_midnight import Word2Vec_SkipGram, ArrayToDatasetForW2V, ConditionalDataset

NameError: name 'nn' is not defined

In [ ]:
batch_count = 103
genres = {'rap', 'rb', 'pop', 'country', 'rock', 'misc'}
print(genres)
print(batch_count)

{'rap', 'pop', 'country', 'rock', 'misc', 'rb'}
103


In [ ]:
total_subset_size = 100000
total_group_size = total_subset_size // len(genres) + 1
max_songs_per_artist = 50

In [ ]:
ds_subset = pd.read_csv(os.path.join('temp', 'genius_lyrics_subset.csv'))
display(ds_subset.head())

,Unnamed: 0,title,tag,artist,year,lyrics,id,language
0,123725,In Memory of W. B. Yeats,misc,W. H. Auden,2013,I\n\nHe disappeared in the dead of winter:\nTh...,134446,en
1,2614099,Sthlm City,pop,Rebecca & Fiona,2018,"[Intro]\nI mean, like, it's okay\nOne day, you...",3931557,en
2,2550228,Fallin 2010,pop,Mozhdah,2010,[Verse 1]\nCan you take it easy\nWhile I make ...,3840000,en
3,3539706,Can the Circle Be Unbroken,country,Johnny Cash,1982,I was standing by the window\nOn one cold and ...,5350667,en
4,2650114,Marijuana,pop,Jah Cure,2019,[Intro: Jah Cure]\nOne puff one draw\nFrees th...,3982786,en


In [ ]:
teal = Teal(ds_subset)

Loaded Cache for Teal._prepare_ds_data.cached
Loaded Cache for Teal._get_custom_tokens.cached
Loaded Cache for Teal._get_genre_dict.cached
Loaded Cache for Teal._prepare_tfidf.cached
Loaded Cache for Midnight._prepare_vocabulary temp\lyrics_sp.model
Loaded Cache for Teal._prepare_embedder.cached


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
dataset = ConditionalDataset(teal)
dataloader = data.DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=dataset.collate_fn,
)